<a href="https://colab.research.google.com/github/hermela-tt/SATC_Dialogue_Analytics/blob/main/SATC_Dialogue_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:

# COMMIT - CREATED USING COLAB, INITIAL PROJECT SETUP AND SATC DATASET EXPLORATION


import pandas as pd

#check what files viewable
import os
os.listdir()

# upload file each time, option
#from google.colab import files
# files.upload()

# Load the CSV directly (bc it's already uploaded to google Colab environment files on the left panel)
df = pd.read_csv('SATC_all_lines.csv')


# Quickly print shape and columns without heavy functions,  no buffering
print("Shape (Rows, Columns):", df.shape)
print("Columns:", list(df.columns))


# Show first 3 rows
display(df.head(3))




# CLEANING PRELIMINARY


# missing values across all columns
print("\n Missing Values Per Column")
print(df.isnull().sum())


# check the top 15 unique speakers to see how characters are named
print("\n Top 15 Speakers by Dialogue Count ")


# Replace 'speaker' with your exact column name if it's capitalized differently (e.g., 'Speaker')
print(df["Speaker"].value_counts().head(15))




# drop null - remove rows where there is no character (speaker) or dialogue (line) to clean into a new dataframe (df_clean)
df_clean = df.dropna(subset=["Speaker", "Line"])


print(" \n Original rows: ", len(df))
print("\n Clean rows: ", len(df_clean))




# WHO TALKS THE MOST


speaker_counts = df_clean["Speaker"].value_counts()


dialogue_share = (speaker_counts / speaker_counts.sum() * 100).round(2)


print(dialogue_share.head(10))




# CHECKING TOTAL ROWS AND VALUES


#checking total number of rows by pulling bottom of dataset
df["Unnamed: 0"].tail()


#why does it reset
df["Unnamed: 0"].nunique()


#how many unique values, top 20 rows
df["Unnamed: 0"].head(20)


# how many unique value
df["Unnamed: 0"].nunique()


print ("\n ")
#fixing name inconsitences


# to stop mistakes - waring: A value is trying to be set on a copy of a slice from a DataFrame (Try using .loc[row_indexer,col_indexer] = value instead)
df_clean = df.dropna(subset=["Speaker", "Line"]).copy()
# remove spaces for speakers
df_clean["Speaker"] = df_clean["Speaker"].str.strip()


#are speaker names consistent
df_clean["Speaker"].unique()[:30]




# BEGIN CHARACTER ANALYSIS


# new dataframe for cleaned speaker column (without spaces), call first 15
df_clean["Speaker"].value_counts().head(15)




# COMMIT - REMOVED OLD CODE


# make sure we have a clean, duplicate copy
df_clean = df.copy()
print("Original rows:", len(df))
print("Clean copy rows:", len(df_clean))

# remove rows missing a speaker and line
df_clean = df_clean.dropna(subset=["Speaker", "Line"])

print("Rows after removing missing dialogue:")
print(len(df_clean))
#check mising or overlap (null) total in columns
df_clean.isnull().sum()

# (standardized) dropped index column (#)
df_clean = df_clean.drop(columns=["Unnamed: 0"])

print(df_clean.columns)


# to get total after cleaning new dataset
df_clean.nunique()

# to check on unique episode numbers
df_clean["Episode"].value_counts().sort_index()

# checking on why episodes end at
df_clean["Speaker"].value_counts().head(20)



# COMMIT - CLEAN SATC DATASET AND ANALYZE CHARACTER FREQUENCY

#created new dataframe for the four main characters (based on highest freq on cleaned dataframe)
main_characters = ["Carrie", "Miranda", "Samantha", "Charlotte"]
# copied clean copy for speakers of 4 main characters
df_main = df_clean[df_clean["Speaker"].isin(main_characters)].copy()
# print title before dataset
print("\n Main character rows:", df_main.shape)
df_main.head()

# get values of who talks the most of the 4
df_main["Speaker"].value_counts()

# vocab of the 4 characters (unqiue dialogue lines)
df_main.groupby("Speaker")["Line"].nunique()

#looking at variety of words
from collections import Counter
import re
#for each character, take and join the speaker text from their lines into a string
for character in ["Carrie", "Miranda", "Samantha", "Charlotte"]:
    text = " ".join(df_main[df_main["Speaker"] == character]["Line"].astype(str))

    # Convert to lowercase and keep only words
    words = re.findall(r"\b[a-z']+\b", text.lower())

    # Show the 20 most common words
    print(f"\n{character}")
    print(Counter(words).most_common(20))

#question - dialogue percentages between the top four --- creating a table to highlight mmy findings
speaker_counts = df_main["Speaker"].value_counts()

speaker_table = (
    speaker_counts
    .rename_axis("Character")
    .reset_index(name="Dialogue Lines")
)

speaker_table["% of Main Four Dialogue"] = (
    speaker_table["Dialogue Lines"] /
    speaker_table["Dialogue Lines"].sum() * 100
).round(2)

speaker_table["% of All Dialogue"] = (
    speaker_table["Dialogue Lines"] /
    len(df_clean) * 100
).round(2)

speaker_table

#finding - Carrie accounts for 35.57% of all dialogue and 52.70% amongst the four protagonists, supporting her main character status.

#question - after removing stop words, what words found for top 4 characters and what do they mean

custom_stopwords = {
    "i'm", "it's", "don't", "just", "know", "like",
    "right", "think", "really", "oh", "got", "did",
    "gonna", "can't", "that's", "you're", "he's",
    "i've", "i'll", "we're", "want", "going"
}

print(custom_stopwords)

# remove standard stop words (the, I, and) to explore actual vocab
from collections import Counter
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

#for 4 personas
for character in ["Carrie", "Miranda", "Samantha", "Charlotte"]:
    #true or false, select each characters lines, join into one string
    text = " ".join(
        df_main[df_main["Speaker"] == character]["Line"].astype(str)
    )
    # make joined string into word boundary's, a-z, allowing for abstrophy's, lower case words
    words = re.findall(r"\b[a-z']+\b", text.lower())
    # drop stop and custom stop words
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS
        and word not in custom_stopwords
    ]
    # count and bring 20 most common remaining words
    common_words = Counter(words).most_common(20)
    # skip a line and print characters name before printing their commmon owrds
    print(f"\n{character}")
    print(common_words)


  # custom stop words to avoid basic words, based on prelimainry dropping of basic english standard stop words
custom_stopwords = {
    "i'm", "it's", "don't", "just", "know", "like",
    "right", "think", "really", "oh", "got", "did",
    "gonna", "can't", "that's", "you're", "he's",
    "i've", "i'll", "we're", "want", "going"
}

# COMMIT - REMOVED MISSING DIALOGUE ROWS, STANDARDIZE SPEAKER NAMES,  FIND TOP 4 SATC CHARACTERS FIND TOP 4 SATC CHRACTERS, AND BEGIN DROPPING STOP WORDS



Shape (Rows, Columns): (39999, 6)
Columns: ['Unnamed: 0', 'Season', 'Episode', 'Speaker', 'Line', 'date_job']


,Unnamed: 0,Season,Episode,Speaker,Line,date_job
0,0,1.0,1.0,Carrie,"Once upon a time, an English journalist came t...",NaN
1,1,1.0,1.0,Carrie,Elizabeth was attractive and bright.,NaN
2,2,1.0,1.0,Carrie,Right away she hooked up with one of the city'...,NaN



 Missing Values Per Column
Unnamed: 0        0
Season           10
Episode          11
Speaker         803
Line            788
date_job      39911
dtype: int64

 Top 15 Speakers by Dialogue Count 
Speaker
Carrie       13941
Miranda       4780
Samantha      4067
Charlotte     3667
Big           1233
Steve          801
Aidan          703
Aleksandr      511
Trey           382
Stanford       367
Jack           364
Smith          222
Anthony        210
Richard        189
Robert         159
Name: count, dtype: int64
 
 Original rows:  39999

 Clean rows:  39195
Speaker
Carrie       35.57
Miranda      12.20
Samantha     10.38
Charlotte     9.36
Big           3.15
Steve         2.04
Aidan         1.79
Aleksandr     1.30
Trey          0.97
Stanford      0.94
Name: count, dtype: float64

 
Original rows: 39999
Clean copy rows: 39999
Rows after removing missing dialogue:
39195
Index(['Season', 'Episode', 'Speaker', 'Line', 'date_job'], dtype='object')

 Main character rows: (26455, 5)

Carrie
[(

In [11]:
import pandas as pd

# Load the SATC dataset from the file that is now visible in Colab
df = pd.read_csv("SATC_all_lines.csv")

# Check that it loaded correctly
print(df.shape)

# Show first 5 rows
df.head()

(39999, 6)


,Unnamed: 0,Season,Episode,Speaker,Line,date_job
0,0,1.0,1.0,Carrie,"Once upon a time, an English journalist came t...",NaN
1,1,1.0,1.0,Carrie,Elizabeth was attractive and bright.,NaN
2,2,1.0,1.0,Carrie,Right away she hooked up with one of the city'...,NaN
3,3,1.0,1.0,Tim,The question remains-- Is this really a compan...,NaN
4,4,1.0,1.0,Carrie,"Tim was 42, a well-liked and respected investm...",NaN


In [12]:
 print(df_clean.shape)

(39195, 5)


In [13]:
print(df_main.shape)

(26455, 5)


In [7]:
import os
os.listdir()

['.config', 'SATC_all_lines.csv', 'sample_data']

In [6]:
from google.colab import files

files.upload()

Saving SATC_all_lines.csv to SATC_all_lines.csv


{'SATC_all_lines.csv': b',Season,Episode,Speaker,Line,date_job\n0,1.0,1.0,Carrie,"Once upon a time, an English journalist came to New York.",\n1,1.0,1.0,Carrie,Elizabeth was attractive and bright.,\n2,1.0,1.0,Carrie,Right away she hooked up with one of the city\'s typically eligible bachelors.,\n3,1.0,1.0,Tim,The question remains-- Is this really a company we want to own? ,\n4,1.0,1.0,Carrie,"Tim was 42, a well-liked and respected investment banker who made about two million a year.",\n5,1.0,1.0,Carrie,"They met one evening, in typical New York fashion at a gallery opening.",\n6,1.0,1.0,Tim,Like it? ,\n7,1.0,1.0,Elizabeth,"Yes, actually. I think it\'s quite interesting. What?",\n8,1.0,1.0,Tim, I feel like I know you from somewhere.,\n9,1.0,1.0,Elizabeth,Doubtful. I only just moved here from London.,\n10,1.0,1.0,Tim,London? Really? That\'s my all-time favorite city.,\n11,1.0,1.0,Elizabeth,- It is?,\n12,1.0,1.0,Tim,Absolutely.,\n13,1.0,1.0,Carrie,It was love at first sight.,\n14,1.0,1.0,